# Story 3: Do stricter gun laws reduce firearm gun deaths?
**Muhammad Suffyan Khan · DATA 608 · Fall 2026**

**Answer in brief:** Across 50 states, stronger 2024 law grades coincide with lower 2024 firearm mortality rates. This single-year state comparison cannot establish that the law differences caused the mortality differences. The outcome includes firearm suicides and other firearm deaths as well as homicides.

**Software change:** R (base graphics, via R/RStudio) produced the heat maps. Stories 1 and 2 used Python visualization tools. The presentation is supplied separately as PowerPoint and PDF.

## 1. Sources and the precise question
- **Mortality:** [CDC NCHS provisional quarterly mortality dataset](https://data.cdc.gov/National-Center-for-Health-Statistics/NCHS-VSRR-Quarterly-provisional-estimates-for-sele/489q-934x). Query its open API for 2024 Q4, **12 months ending with quarter**, **Firearm-related injury**, **Age-adjusted**. This yields deaths per 100,000 for 50 states. The complete successful response and requested URL are saved under `../data/raw/`. The values for New York, Mississippi, California, and Massachusetts match the one-decimal figures on the [current CDC firearm mortality page](https://www.cdc.gov/nchs/state-stats/deaths/firearms.html).
- **Law strength:** [GIFFORDS 2024 Annual Gun Law Scorecard](https://giffords.org/resources/scorecard2024/). Extract **only** its 50 state law grades and law-strength ranks. Its own mortality column is excluded. The source is a gun safety advocacy organization and its grading reflects its policy choices.

The question is whether 2024 state law grades and firearm mortality rates show a pattern. An association in one cross-section cannot tell us whether changing a law reduces deaths.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'data/processed/state_analysis_2024.csv').exists(), 'Open the story folder as your working directory'
raw = json.loads((ROOT/'data/raw/CDC_API_2024_firearm_age_adjusted.json').read_text())
response = raw['response']
print('CDC endpoint:', raw['request_url'])
print('Records:', len(response), '| period:', response[0]['time_period'], '| cause:', response[0]['cause_of_death'], '| rate:', response[0]['rate_type'])

CDC endpoint: https://data.cdc.gov/resource/489q-934x.json?%24where=year_and_quarter%3D%272024+Q4%27+and+time_period%3D%2712+months+ending+with+quarter%27+and+cause_of_death%3D%27Firearm-related+injury%27+and+rate_type%3D%27Age-adjusted%27&%24limit=10
Records: 1 | period: 12 months ending with quarter | cause: Firearm-related injury | rate: Age-adjusted


### How to reproduce the API call
The preserved record above comes from a live CDC Socrata API request, not a hand-entered table. The script `../scripts/fetch_and_prepare.py` performs the query and verifies a single response before rebuilding all state data. Rerunning it may obtain later revised CDC values; the original successful JSON remains archived for this analysis.

In [2]:
# Optional: direct live call (saved response is sufficient for the packaged analysis)
# import urllib.request
# live_response = json.load(urllib.request.urlopen(raw['request_url'], timeout=45))
# assert len(live_response) == 1
# print(live_response[0]['rate_new_york'])

## 2. Five-level ordinal law scale
Collapse the **first letter** of each GIFFORDS grade into five ordered categories; keep the uncollapsed A-, B+, etc. in the CSV. This is an ordinal **Likert-style classification** made for the assignment, rather than an opinion survey or a standardized numeric distance between grades.

| Level | Label | Original grades | States |
| ---: | --- | --- | ---: |
| 1 | Most lax | F | 22 |
| 2 | Lax | D+, D, D- | 3 |
| 3 | Middle | C+, C, C- | 5 |
| 4 | Strict | B+, B, B- | 7 |
| 5 | Strictest | A+, A, A- | 13 |

This preserves source-defined ordering. Group sizes vary because these are grade bins, not equal-state quintiles. The complete assignment for each state is in `../data/processed/law_scale_50_states.csv`.

In [3]:
state = pd.read_csv(ROOT/'data/processed/state_analysis_2024.csv')
assert len(state)==50 and state.state.nunique()==50 and state.abbr.nunique()==50
assert state.law_level.notna().all() and state.firearm_death_rate_2024.notna().all()
print(state.groupby(['law_level','gun_law_grade']).size().to_string())
print('State count:', len(state), '| missing values:', int(state[['law_level','firearm_death_rate_2024']].isna().sum().sum()))

law_level  gun_law_grade
1          F                22
2          D-                3
3          C                 1
           C+                1
           C-                3
4          B                 2
           B+                1
           B-                4
5          A                 5
           A-                8
State count: 50 | missing values: 0


## 3. Heat maps and visual design
**Why these visuals?** Two equal-tile schematic maps put every state at comparable visual size. The law map emphasizes the five ordinal categories. The mortality map prints each age-adjusted rate and uses a fixed 0-to-30 sequential scale, so readers can locate both broad regional patterns and individual outliers. The state maps now include direct takeaways, printed law levels, and numbered outlines linking Hawaii, Mississippi and New Mexico to their annotations. The mortality-band heat map prints state counts and within-group percentages, with color encoding the percentage on a fixed 0-to-100 scale. Black outlines emphasize the F group at 16 or more deaths (15/22, 68%) and the A group below 8 (7/13, 54%). This makes the message visible on the chart and accounts for unequal group sizes. Rounded cell percentages may not sum to 100%. Base R produced every chart with palettes from `hcl.colors` chosen for readable contrast. The schematic tiles are not geographic area maps.

### Law strength heat map
![50-state law strength tile map](../figures/01_law_strength_heatmap.png)

### CDC firearm mortality heat map
![50-state mortality tile map](../figures/02_firearm_mortality_heatmap.png)

### State counts by law category and mortality band
![Law category by rate band heat map](../figures/03_law_by_mortality_count_heatmap.png)

## 4. Numeric check and answer

In [4]:
summary = pd.read_csv(ROOT/'data/processed/five_level_summary.csv')
print(summary[['law_level','law_label','states','mean_rate','median_rate','min_rate','max_rate']].to_string(index=False))
print('Spearman rho:', round(state[['law_level','firearm_death_rate_2024']].corr(method='spearman').iloc[0,1],3))

 law_level        law_label  states  mean_rate  median_rate  min_rate  max_rate
         1  1  Most lax (F)      22     18.595         18.8      11.8      28.0
         2       2  Lax (D)       3     14.000         14.6      10.2      17.2
         3    3  Middle (C)       5     12.780         12.1      10.9      16.3
         4    4  Strict (B)       7     14.400         11.9       9.8      26.6
         5 5  Strictest (A)      13      8.431          7.0       3.7      15.3
Spearman rho: -0.717


In [5]:
bands = pd.read_csv(ROOT/'data/processed/mortality_band_counts_and_shares.csv')
assert len(bands) == 25 and bands.state_count.sum() == 50
print(bands.to_string(index=False))
f = state[state.law_level == 1]
a = state[state.law_level == 5]
print(f'Most lax at 16 or more: {(f.firearm_death_rate_2024 >= 16).sum()}/{len(f)} = {100*(f.firearm_death_rate_2024 >= 16).mean():.1f}%')
print(f'Strictest below 8: {(a.firearm_death_rate_2024 < 8).sum()}/{len(a)} = {100*(a.firearm_death_rate_2024 < 8).mean():.1f}%')

 law_level mortality_band  state_count  group_states  percent_of_group
         1          [0,8)            0            22          0.000000
         2          [0,8)            0             3          0.000000
         3          [0,8)            0             5          0.000000
         4          [0,8)            0             7          0.000000
         5          [0,8)            7            13         53.846154
         1         [8,12)            1            22          4.545455
         2         [8,12)            1             3         33.333333
         3         [8,12)            2             5         40.000000
         4         [8,12)            4             7         57.142857
         5         [8,12)            3            13         23.076923
         1        [12,16)            6            22         27.272727
         2        [12,16)            1             3         33.333333
         3        [12,16)            2             5         40.000000
      

The 13 A-grade states averaged **8.4 deaths per 100,000**, compared with **18.6** among the 22 F-grade states. Their medians were **7.0** and **18.8**. The ordinal law-level versus state mortality Spearman coefficient is **-0.717** (exploratory two-sided p = 4.72 × 10⁻⁹), which describes a strong inverse state-level pattern. This is descriptive; do not read the mean difference as an estimated effect of legislation.

The pattern has important exceptions. **New Mexico** received a B- law grade but had **26.6** deaths per 100,000, the second highest of the 50 states. The small B group therefore has a mean of **14.4**, above the C group's **12.8**, although its median (**11.9**) is a little lower than C's (**12.1**). This is why the visual story should show individual states and group sizes instead of only five category averages.

## 5. Limits, visual-choice explanation, and conclusion
- The dataset measures **all firearm-related injury deaths**. Suicide, homicide, and other intents can follow different dynamics. Mortality is not identical to the assignment's informal phrase “gun violence deaths.”
- Law grades are from a 2024 policy scorecard; the deaths are from the same year. Law adoption and effects can take time, and contemporaneous measures do not establish temporal ordering.
- Gun ownership, poverty, state demographics, geography, access to care, enforcement, neighboring laws, and other factors were not controlled. A state-level association cannot be assigned to individuals.
- GIFFORDS grades are one advocacy group's policy index, and collapsing +/– grades discards information. The five groups are uneven. The CDC API dataset is designated **provisional** and can change.
- Each state has equal weight in the category summary. The averages of state rates do not equal national rates. A small statistical p value does not remove confounding.

**Conclusion:** States with stronger 2024 gun law grades tended to report lower age-adjusted firearm mortality in 2024. These heat maps support an association, including visible exceptions, but cannot determine whether stricter laws **resulted in** fewer deaths. A stronger causal study would use policy implementation dates, repeated years, and appropriate comparison states and control variables.

**Course rubric connection:** The story addresses the exact question (focus), uses auditable API data and 50 explicit state assignments (fidelity), reduces the visual grammar to accessible maps and legible scales (simplicity and saliency), names actionable questions for future investigation (utility), gives software and source details (efficacy), and uses consistent labels and typography across charts and slides (uniformity and amity).

## Sources
See `../sources/SOURCES.md` for the exact API query, archival methods, definitions, visual palette, and limitations. The archived GIFFORDS HTML preserves the grades as retrieved. The raw CDC JSON preserves the complete successful request and response.
